# Memory Management

Memory allows agents to retain information across interactions. **Short-term memory** holds the current conversation context; **long-term memory** persists facts, preferences, and knowledge across sessions.

## Implementation with Flyte v2

This notebook reimplements the LangChain `ConversationBufferMemory` and LangGraph `InMemoryStore` patterns using **Flyte v2 primitives only**.

#### LangChain / LangGraph vs Flyte v2 — Key Differences

| Aspect | LangChain / LangGraph | Flyte v2 |
|--------|----------------------|----------|
| **Short-term memory** | `ConversationBufferMemory` (in-process, lost on restart) | Typed `ConversationMemory` dataclass — serialized by Flyte, survives retries |
| **Long-term memory** | LangGraph `InMemoryStore` (per-process, ephemeral) | Typed `UserProfile` dataclass passed via Flyte object store — durable |
| **Memory scope** | In-process only | Task outputs are stored in object storage; any downstream task can access them |
| **Observability** | Log parsing | Full memory state visible in Flyte UI as structured task output |
| **Serialization** | Automatic but opaque | Explicit typed dataclasses — inspectable, versionable |
| **Secrets** | `.env` / `os.environ` | `flyte.Secret` injected by cluster (no plaintext in code) |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' anthropic

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-ant-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import timedelta

import anthropic
import flyte

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="memory-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.40.0")
)

memory_env = flyte.TaskEnvironment(
    name="memory_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define the data models

**Short-term memory** (`ConversationMemory`) holds the current conversation thread.
**Long-term memory** (`UserProfile`) accumulates facts about the user across sessions.

Both are immutable-style dataclasses: `append()` / `update()` return **new** objects. This matches Flyte's data-lineage model — each call's memory state is a distinct, traceable snapshot.

Compare to LangChain:
- `ConversationBufferMemory.save_context(...)` mutates in-place and is lost on pod restart.
- `InMemoryStore.put(...)` stores per-process, not across tasks or retries.

With Flyte, both memory objects are serialized to object storage automatically and can be passed to any downstream task.

In [ ]:
@dataclass
class Message:
    role: str
    content: str


@dataclass
class ConversationMemory:
    """Short-term: current conversation thread. Replaces LangChain ConversationBufferMemory."""
    messages: list[Message] = field(default_factory=list)

    def append(self, role: str, content: str) -> ConversationMemory:
        return ConversationMemory(messages=self.messages + [Message(role, content)])

    def to_api_format(self) -> list[dict]:
        return [{"role": m.role, "content": m.content} for m in self.messages]


@dataclass
class UserProfile:
    """Long-term: user facts that persist across sessions. Replaces LangGraph InMemoryStore."""
    name: str = ""
    preferences: dict = field(default_factory=dict)
    visited_places: list[str] = field(default_factory=list)

    def remember(self, key: str, value: str) -> UserProfile:
        return UserProfile(
            name=self.name,
            preferences={**self.preferences, key: value},
            visited_places=self.visited_places,
        )

    def add_visit(self, place: str) -> UserProfile:
        return UserProfile(
            name=self.name,
            preferences=self.preferences,
            visited_places=self.visited_places + [place],
        )

    def summary(self) -> str:
        if not self.preferences and not self.visited_places:
            return "No user profile information available."
        parts = []
        if self.name:
            parts.append(f"User: {self.name}")
        if self.preferences:
            parts.append("Preferences: " + ", ".join(f"{k}={v}" for k, v in self.preferences.items()))
        if self.visited_places:
            parts.append("Previously visited: " + ", ".join(self.visited_places))
        return ". ".join(parts)


@dataclass
class ChatResult:
    """Typed output of one chat turn — response text plus updated memory."""
    response: str
    history: ConversationMemory
    profile: UserProfile

### 5. Define the travel agent task

The task uses `@flyte.trace` to checkpoint each LLM call. On pod failure, Flyte retries from the last checkpoint rather than replaying the entire conversation.

The agent receives both memory objects as typed inputs. It:
1. Builds a system prompt incorporating the long-term `UserProfile`.
2. Sends the short-term `ConversationMemory` as the message history.
3. Returns updated versions of both — the new response appended to history, and any new facts extracted into the profile.

**Why this beats LangChain's approach:** LangChain's `ConversationBufferMemory` is stateful in-process. If the chain is re-instantiated (e.g., after a pod restart), memory is gone. With Flyte, the memory objects are passed as task inputs/outputs and stored in object storage — they survive any failure.

In [ ]:
TRAVEL_AGENT_SYSTEM = """\
You are a friendly travel agent with excellent memory. Help the user plan trips.
When the user mentions a preference (budget, travel style, food, etc.), acknowledge it.
When they mention a place they've visited, acknowledge it.
Keep responses concise (2-3 sentences max)."""

PROFILE_EXTRACTOR_SYSTEM = """\
Extract structured facts from this conversation turn.
Respond ONLY with JSON in this exact format:
{"name": "<user name if mentioned, else empty string>",
 "preference_key": "<one preference key if any, else empty string>",
 "preference_value": "<preference value, else empty string>",
 "visited_place": "<place name if user mentioned visiting, else empty string>"}"""


@flyte.trace
async def _chat(messages: list[dict], system: str) -> str:
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        system=system,
        messages=messages,
    )
    return response.content[0].text


@flyte.trace
async def _extract_facts(user_message: str, agent_response: str) -> dict:
    import json
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        system=PROFILE_EXTRACTOR_SYSTEM,
        messages=[{
            "role": "user",
            "content": f"User said: {user_message}\nAgent replied: {agent_response}",
        }],
    )
    try:
        return json.loads(response.content[0].text)
    except Exception:
        return {}


@memory_env.task(
    retries=2,
    timeout=timedelta(minutes=5),
    cache=flyte.Cache(behavior="disable"),
)
async def chat_turn(
    user_message: str,
    history: ConversationMemory,
    profile: UserProfile,
) -> ChatResult:
    """
    One conversation turn with memory.

    Short-term memory flow:
      1. Build API messages from existing history.
      2. Append user message.
      3. Call agent, get response.
      4. Append response to history.

    Long-term memory flow:
      1. Extract facts from this turn.
      2. Update UserProfile with new facts.
      3. Return updated profile alongside history.
    """
    profile_context = profile.summary()
    system = TRAVEL_AGENT_SYSTEM
    if profile_context != "No user profile information available.":
        system += f"\n\nKnown user context: {profile_context}"

    messages = history.to_api_format() + [{"role": "user", "content": user_message}]

    response = await _chat(messages=messages, system=system)

    facts = await _extract_facts(user_message=user_message, agent_response=response)

    updated_history = history.append("user", user_message).append("assistant", response)

    updated_profile = profile
    if facts.get("name"):
        updated_profile = UserProfile(
            name=facts["name"],
            preferences=updated_profile.preferences,
            visited_places=updated_profile.visited_places,
        )
    if facts.get("preference_key") and facts.get("preference_value"):
        updated_profile = updated_profile.remember(
            facts["preference_key"], facts["preference_value"]
        )
    if facts.get("visited_place"):
        updated_profile = updated_profile.add_visit(facts["visited_place"])

    return ChatResult(response=response, history=updated_history, profile=updated_profile)

### 6. Run a multi-turn conversation locally

Each `flyte.run()` call receives and returns the full memory state. The updated memory from one turn is fed into the next — simulating the `ConversationBufferMemory` pattern, but durably via typed dataclasses.

In [ ]:
history = ConversationMemory()
profile = UserProfile()

CONVERSATION = [
    "Hi! I'm Alex. I love budget travel and hate cold weather.",
    "I visited Bali last year and it was amazing. Where should I go next?",
    "Do you remember my name and what I said about the weather?",
]

for user_msg in CONVERSATION:
    print(f"User: {user_msg}")
    run = flyte.run(chat_turn, user_message=user_msg, history=history, profile=profile)
    run.wait()
    result: ChatResult = run.outputs()[0]
    print(f"Agent: {result.response}")
    print(f"Profile now: {result.profile.summary()}")
    print()
    history = result.history
    profile = result.profile

### Running remotely

Switch to remote execution to get full Flyte observability: the `ConversationMemory` and `UserProfile` objects appear as structured outputs in the UI, making memory state inspectable without log parsing.

In [ ]:
# Same API — Flyte handles local vs remote transparently via .flyte/config.yaml
run = flyte.run(
    chat_turn,
    user_message="I prefer eco-friendly accommodations. Any ideas for Southeast Asia?",
    history=history,
    profile=profile,
)
run.wait()
result = run.outputs()[0]
print(result.response)
print("Updated profile:", result.profile.summary())

## Scaling the pattern

For high-throughput conversational systems, the `ReusePolicy` eliminates cold-start cost per turn. Each pod handles multiple concurrent conversations, with the `ConversationMemory` object passed in per-request rather than stored globally (no shared mutable state between requests).

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
# Production: handle many concurrent chat sessions
production_memory_env = flyte.TaskEnvironment(
    name="memory_agent_prod",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
    reusable=flyte.ReusePolicy(
        replicas=(2, 8),
        concurrency=16,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=15),
    ),
)